# Dynamical Batching vs. Uniform Batching on the spiral — minimal reproduction

This notebook reproduces, for **one seed**, the core experiment:

1. Generate the **spiral** dataset (train + test).
2. Build an **MLP** (JAX).
3. Train it twice from the **same initialisation and seed**:
   * **UB** — Uniform Batching ($A = 1$: every batch is class-balanced),
   * **DB** — Dynamical Batching ($A = 20$, $T = 320$ steps per oscillation).
4. Compute the three **weight-level switching measures** once per cycle:
   switching-set size $|\mathcal S|/n$, switching-set persistence $J(\mathcal S_c, \mathcal S_{c-1})$,
   and the pair-averaged switching distance $d_{ij}$.
5. Save one CSV per run and draw the figures.

The notebook is fully self-contained: use Colab to run all cells.

The **entire training run is a single compiled XLA program**: the steps inside an evaluation block are a
`jax.lax.scan`, the blocks themselves are an outer `lax.scan`, and batches are sampled on-device.
For the switching measures only the per-class mean parameters over the central window of each oscillation are
accumulated, on-device. UB and DB share **one** compilation, because the amplitude $A$ is a runtime argument.

> Runtime → *Change runtime type* → CPU is perfectly fine (the network is tiny); a GPU works too.

## 0 · Setup
Colab already ships `jax`, `optax`, `numpy`, `pandas` and `matplotlib`; the cell below only installs `optax` if it is missing.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("optax") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optax"], check=True)

import time
from math import sqrt
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import optax
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from jax.flatten_util import ravel_pytree

print("JAX", jax.__version__, "| devices:", jax.devices())

## 1 · Configuration
All knobs live in this one cell.

**Protocol.** Classes take turns being the *focus class*: each **oscillation** lasts `period_length` $=T$ steps and
belongs to one class (focus class $=\lfloor t/T\rfloor \bmod C$). During its oscillation, the focus weight ramps linearly
$1 \to A \to 1$ (triangle, peak at $T/2$); the batch composition is $p_c \propto w_c$ with $w_{\text{focus}}$ the ramp
and $w_{\text{other}} = 1$, and the per-class counts are $\lfloor p_c B\rfloor$.
A **cycle** is $C$ consecutive oscillations, one per class, i.e. $C\,T$ steps.

Oscillations run for the whole training. The amplitude $A$ is the only difference between the two runs:
$A = 1$ gives flat weights, i.e. uniform batching.

In [ ]:
CONFIG = {
    "seed": 0,                       # model init + batch sampling; SAME for UB and DB
    "data": {
        "num_classes": 3,
        "points_per_class": 100,
        "revolutions": 4.0,          # angle span = revolutions * pi
        "noise_std": 0.2,            # angular noise (rad)
        "min_radius": 0.01,
        "angular_offsets": None,     # e.g. [0, 120, 240] (degrees); None = evenly spaced
        "randomize_offsets": False,
        "random_seed": 10,            # train set uses this seed, test set uses seed + 1
    },
    "model": {
        "architecture": "mlp_relu",  # "mlp" (linear), "mlp_relu", "mlp_tanh"
        "nn_width": 50,
        "num_hidden_layers": 1,
        "use_bias": True,
        "init_type": "glorot",       # "glorot" or "lecun"
    },
    "training": {
        "training_steps": 125_000,
        "batch_size": 50,
        "save_metrics_every_n_steps": 40,
    },
    "optimizer": {
        "optimizer_type": "adam",    # "adam" or "sgd"
        "learning_rate": 2e-3,
        "beta1": 0.9, "beta2": 0.999, "eps": 1e-8,
        "momentum": 0.9,             # only used by sgd
        "weight_decay": 0.0,         # L2 penalty 0.5*wd*||theta||^2 added to the gradient
    },
    "oscillations": {
        "period_length": 320,        # T: steps per oscillation (one class)
    },
    "analysis": {
        "switching_energy_quantile": 0.8,   # q of the energy set S
    },
    "runs": {"UB": 1.0, "DB": 20.0},        # amplitude A of each run; everything else is shared
    "output_dir": "results",
}

OUT = Path(CONFIG["output_dir"]); OUT.mkdir(parents=True, exist_ok=True)
C_ = CONFIG["data"]["num_classes"]; T_ = CONFIG["oscillations"]["period_length"]
E_ = CONFIG["training"]["save_metrics_every_n_steps"]; S_ = CONFIG["training"]["training_steps"]
assert S_ % E_ == 0, "training_steps must be a multiple of save_metrics_every_n_steps"
assert T_ >= 4, "period_length must be >= 4 (central window [T/4, 3T/4))"
print(f"cycle length C*T = {C_*T_} steps -> {S_ // (C_*T_)} complete cycles; "
      f"{S_ // E_} evaluations; cycle ends aligned with evaluations: {(C_*T_) % E_ == 0}")

## 2 · Spiral dataset
Class $j$ is the arm $r\in[r_{\min},1]$, $\theta \in [\phi_j, \phi_j + \text{revolutions}\cdot\pi]$, with Gaussian noise
on $\theta$. The test set is drawn from the same distribution with a different seed.

In [ ]:
def generate_spiral_data(points_per_class, num_classes, revolutions, noise_std, random_seed,
                         angular_offsets=None, randomize_offsets=False, min_radius=0.05):
    rng = np.random.default_rng(random_seed)
    n, c = points_per_class, num_classes
    if angular_offsets is not None:
        offsets = np.deg2rad(np.array(angular_offsets, dtype=np.float32))
    elif randomize_offsets:
        offsets = rng.uniform(0.0, 2.0 * np.pi, size=c).astype(np.float32)
    else:
        offsets = np.array([2.0 * np.pi * j / c for j in range(c)], dtype=np.float32)
    x_all = np.zeros((n * c, 2), dtype=np.float32)
    y_all = np.zeros((n * c,), dtype=np.int64)
    for j in range(c):
        ix = slice(n * j, n * (j + 1))
        r = np.linspace(min_radius, 1.0, n, dtype=np.float32)
        theta = np.linspace(offsets[j], offsets[j] + revolutions * np.pi, n, dtype=np.float32)
        theta += rng.normal(0.0, noise_std, size=n).astype(np.float32)
        x_all[ix, 0] = r * np.cos(theta)
        x_all[ix, 1] = r * np.sin(theta)
        y_all[ix] = j
    return x_all, y_all


d = CONFIG["data"]
spiral_kw = dict(points_per_class=d["points_per_class"], num_classes=d["num_classes"],
                 revolutions=d["revolutions"], noise_std=d["noise_std"],
                 angular_offsets=d["angular_offsets"], randomize_offsets=d["randomize_offsets"],
                 min_radius=d["min_radius"])
x_train, y_train = generate_spiral_data(random_seed=d["random_seed"], **spiral_kw)
x_test,  y_test  = generate_spiral_data(random_seed=d["random_seed"] + 1, **spiral_kw)
print("train:", x_train.shape, np.bincount(y_train), "| test:", x_test.shape, np.bincount(y_test))

## 3 · Model
A plain MLP: hidden layers with the chosen activation, then a linear classifier.
Initialisation is Glorot-uniform with zero bias (default) or LeCun-uniform for weights and biases.
With the same seed, both runs start from **identical** weights.

In [ ]:
def linear_init(key, in_dim, out_dim, init_type):
    k_w, k_b = jax.random.split(key)
    if init_type == "glorot":
        bound = sqrt(6.0 / float(in_dim + out_dim))
        return {"kernel": jax.random.uniform(k_w, (in_dim, out_dim), jnp.float32, -bound, bound),
                "bias": jnp.zeros((out_dim,), jnp.float32)}
    bound = 1.0 / sqrt(float(max(1, in_dim)))                      # LeCun uniform
    return {"kernel": jax.random.uniform(k_w, (in_dim, out_dim), jnp.float32, -bound, bound),
            "bias": jax.random.uniform(k_b, (out_dim,), jnp.float32, -bound, bound)}


def build_model(model_cfg, input_dim, num_classes, seed):
    act = {"mlp": lambda z: z, "mlp_relu": jax.nn.relu, "mlp_tanh": jnp.tanh}[model_cfg["architecture"]]
    init_type = str(model_cfg["init_type"]).lower().replace("_", "")
    init_type = "glorot" if init_type in {"glorot", "xavier"} else "lecun"
    use_bias = model_cfg["use_bias"]
    dims = [input_dim] + [model_cfg["nn_width"]] * model_cfg["num_hidden_layers"] + [num_classes]
    keys = jax.random.split(jax.random.PRNGKey(seed), num=max(1, len(dims) - 1))
    params = {"hidden_layers": [linear_init(keys[i], dims[i], dims[i + 1], init_type) for i in range(len(dims) - 2)],
              "classifier": linear_init(keys[len(dims) - 2], dims[-2], dims[-1], init_type)}

    def apply(p, x):
        h = x
        for layer in p["hidden_layers"]:
            h = act(h @ layer["kernel"] + (layer["bias"] if use_bias else 0.0))
        return h @ p["classifier"]["kernel"] + (p["classifier"]["bias"] if use_bias else 0.0)

    def l2(p):   # kernels, plus biases when they are used
        leaves = [l["kernel"] for l in p["hidden_layers"]] + [p["classifier"]["kernel"]]
        if use_bias:
            leaves += [l["bias"] for l in p["hidden_layers"]] + [p["classifier"]["bias"]]
        return sum(jnp.sum(w * w) for w in leaves)

    return params, apply, l2


params0, apply_fn, l2_fn = build_model(CONFIG["model"], 2, C_, CONFIG["seed"])
n_params = ravel_pytree(params0)[0].size
print(f"{CONFIG['model']['architecture']}  width={CONFIG['model']['nn_width']}  "
      f"hidden layers={CONFIG['model']['num_hidden_layers']}  ->  n = {n_params} parameters")

## 4 · Whole training run as one compiled function

`make_trainer` returns `train(A, key)`, a jitted function that runs **all** `training_steps` and returns

* the evaluation log (one entry every `save_metrics_every_n_steps` steps): train/test loss and accuracy on the **full**
  sets, and the oscillation state at the last step of the block (focus class, focus weight, phase within the oscillation);
* the switching accumulators: for every cycle $c$ and class $i$, the sum (and count) of the flattened parameters
  $\theta_{t+1}$ over the central window $t \bmod T \in [T/4, 3T/4)$ of class $i$'s oscillation. Their ratio is the
  class state $\theta_i(c)$.

Implementation notes:
* per-class counts are $\lfloor p_c B\rfloor$, so the batch can be slightly smaller than $B$; the loss is the mean over
  the *actual* batch (masked positions are ignored);
* examples are drawn on-device with `jax.random`, with replacement inside each class.

In [ ]:
def make_trainer(cfg, params0, apply_fn, l2_fn, x_tr, y_tr, x_te, y_te):
    C = cfg["data"]["num_classes"]
    P = cfg["oscillations"]["period_length"]
    B = cfg["training"]["batch_size"]
    total = cfg["training"]["training_steps"]
    E = cfg["training"]["save_metrics_every_n_steps"]
    wd = float(cfg["optimizer"]["weight_decay"])
    cyc_len = P * C
    n_cyc = total // cyc_len + 1          # +1 holds the (discarded) incomplete last cycle
    n_chunks = total // E

    o = cfg["optimizer"]
    if o["optimizer_type"] == "adam":
        opt = optax.adam(o["learning_rate"], b1=o["beta1"], b2=o["beta2"], eps=o["eps"])
    else:
        opt = optax.sgd(o["learning_rate"], momentum=o["momentum"] or None)

    x_tr, x_te = jnp.asarray(x_tr), jnp.asarray(x_te)
    y_tr, y_te = jnp.asarray(y_tr, jnp.int32), jnp.asarray(y_te, jnp.int32)
    # class -> indices table (padded) for per-class sampling with replacement
    per_class = [np.where(np.asarray(y_tr) == c)[0] for c in range(C)]
    n_per = jnp.asarray([len(ix) for ix in per_class], jnp.int32)
    table = np.zeros((C, max(len(ix) for ix in per_class)), np.int32)
    for c, ix in enumerate(per_class):
        table[c, :len(ix)] = ix
    table = jnp.asarray(table)
    ravel = lambda p: ravel_pytree(p)[0]
    n = ravel(params0).size

    def loss_fn(p, x, y, valid):
        ce = optax.softmax_cross_entropy_with_integer_labels(apply_fn(p, x), y)
        loss = jnp.sum(ce * valid) / jnp.sum(valid)
        return loss + (0.5 * wd * l2_fn(p) if wd > 0 else 0.0)

    def evaluate(p, x, y):
        logits = apply_fn(p, x)
        loss = jnp.mean(optax.softmax_cross_entropy_with_integer_labels(logits, y))
        return loss, jnp.mean(jnp.argmax(logits, 1) == y)

    def one_step(carry, step, A):
        params, opt_state, key, th_sum, win_n = carry
        focus = (step // P) % C
        ps = step % P
        # --- focus weight: triangle 1 -> A -> 1 over the oscillation (flat for A = 1)
        slope = 2.0 * (A - 1.0) / P
        fw = jnp.where(ps < P / 2.0, 1.0 + ps * slope, 2.0 * A - ps * slope - 1.0)
        w = jnp.ones(C).at[focus].set(fw)
        counts = jnp.floor(w / w.sum() * B).astype(jnp.int32)
        # --- sample the batch: position j belongs to class searchsorted(cumsum(counts), j)
        key, k1 = jax.random.split(key)
        cum = jnp.cumsum(counts)
        pos = jnp.arange(B)
        cls = jnp.minimum(jnp.searchsorted(cum, pos, side="right"), C - 1)
        valid = (pos < cum[-1]).astype(jnp.float32)
        within = jax.random.randint(k1, (B,), 0, n_per[cls])
        idx = table[cls, within]
        # --- optimiser step
        grads = jax.grad(loss_fn)(params, x_tr[idx], y_tr[idx], valid)
        updates, opt_state = opt.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        # --- switching bookkeeping (central window of the focus-class oscillation)
        cyc = step // cyc_len
        in_win = (ps >= P // 4) & (ps < (3 * P) // 4)
        th_sum = th_sum.at[cyc, focus].add(jnp.where(in_win, ravel(params), 0.0))
        win_n = win_n.at[cyc, focus].add(in_win.astype(jnp.int32))
        return (params, opt_state, key, th_sum, win_n), (focus, fw, ps / P)

    def one_chunk(carry, chunk, A):
        steps = chunk * E + jnp.arange(E)
        carry, osc = jax.lax.scan(lambda c, s: one_step(c, s, A), carry, steps)
        params = carry[0]
        tr_loss, tr_acc = evaluate(params, x_tr, y_tr)
        te_loss, te_acc = evaluate(params, x_te, y_te)
        log = dict(step=(chunk + 1) * E, train_loss=tr_loss, train_accuracy=tr_acc, test_loss=te_loss,
                   test_accuracy=te_acc, focus_class=osc[0][-1], focus_weight=osc[1][-1], osc_phase=osc[2][-1])
        return carry, log

    @jax.jit
    def train(A, seed_key):
        carry = (params0, opt.init(params0), seed_key,
                 jnp.zeros((n_cyc, C, n), jnp.float32), jnp.zeros((n_cyc, C), jnp.int32))
        carry, log = jax.lax.scan(lambda c, k: one_chunk(c, k, A), carry, jnp.arange(n_chunks))
        params, _, _, th_sum, win_n = carry
        return params, log, th_sum, win_n

    return train


train = make_trainer(CONFIG, params0, apply_fn, l2_fn, x_train, y_train, x_test, y_test)

## 5 · Switching measures (per cycle)
For cycle $c$ with class states $\theta_i(c)$ (rows in cycle order, class position $p_i$):

* **drift** $D(c) = m(c) - m(c-1)$, with $m(c)$ the cycle mean (not available in the first cycle);
* **oscillation residual** $u_{i,k} = \theta_{i,k} - m_k - D_k\,(p_i - \tfrac{C-1}{2})/C$;
* **energy** $a_k^2 = \tfrac1C\sum_i u_{i,k}^2$; the **switching set** $\mathcal S$ is the smallest set of weights carrying a
  fraction $q=0.8$ of the total energy → **switching-set size** $|\mathcal S|/n$;
* **persistence** $J(\mathcal S_c,\mathcal S_{c-1})$ (Jaccard overlap between consecutive cycles);
* **switching distance** $d_{ij} = \lVert\theta_i-\theta_j\rVert_2$, averaged over the class pairs.

Cycles with a missing class window are skipped, and the next cycle is then treated like a first cycle
(no drift removed, no persistence).

In [ ]:
def class_pairs(C):   # adjacent pairs first: (0,1), (1,2), ..., then the rest
    return [(l, l + 1) for l in range(C - 1)] + [(l, r) for l in range(C) for r in range(l + 2, C)]


def energy_set(energy, q):
    # boolean mask of the fewest entries whose energy sums to >= q * total
    total = float(energy.sum())
    mask = np.zeros(energy.size, dtype=bool)
    if total <= 1e-12:
        return mask
    order = np.argsort(energy)[::-1]
    m = int(np.searchsorted(np.cumsum(energy[order]), q * total)) + 1
    mask[order[:min(m, energy.size)]] = True
    return mask


def switching_table(th_sum, win_n, cfg):
    C = cfg["data"]["num_classes"]; P = cfg["oscillations"]["period_length"]
    q = cfg["analysis"]["switching_energy_quantile"]
    cyc_len, n_full = C * P, cfg["training"]["training_steps"] // (C * P)
    pairs = class_pairs(C)
    pc = np.arange(C, dtype=np.float64) - (C - 1) / 2.0
    th_sum = np.asarray(th_sum, np.float64); win_n = np.asarray(win_n)

    rows, prev_theta, prev_cycle, prev_mask = [], None, None, None
    for c in range(n_full):
        if np.any(win_n[c] == 0):
            print(f"cycle {c}: missing class windows, skipped"); continue
        theta = th_sum[c] / win_n[c][:, None]                 # (C, n), central-window means
        linked = prev_cycle == c - 1
        mean_now = theta.mean(0)
        U = theta - mean_now
        if linked:
            U -= np.outer(pc / C, mean_now - prev_theta.mean(0))
        S = energy_set(np.mean(U * U, axis=0), q)
        persist = np.nan
        if linked:
            union = np.count_nonzero(S | prev_mask)
            persist = np.count_nonzero(S & prev_mask) / union if union else np.nan
        d = np.mean([np.linalg.norm(theta[i] - theta[j]) for i, j in pairs])
        rows.append(dict(cycle=c, cycle_end_step=(c + 1) * cyc_len,
                         switch_frac=S.mean(), switch_persist=persist, switching_distance=d))
        prev_theta, prev_cycle, prev_mask = theta, c, S
    return pd.DataFrame(rows)

## 6 · Train UB and DB, save one CSV per run
The first call compiles (a few seconds); the second one reuses the compiled program.

Each CSV has one row per evaluation step with: `step`, train/test accuracy and loss, the oscillation state
(`focus_class`, `focus_weight`, `osc_phase`), and the three switching measures with the `cycle` they belong to.
The switching measures sit on the row where that cycle closed (the first logged step ≥ cycle end); other rows are empty.

In [ ]:
RESULTS = {}
for name, A in CONFIG["runs"].items():
    t0 = time.time()
    params, log, th_sum, win_n = jax.block_until_ready(train(jnp.float32(A), jax.random.PRNGKey(CONFIG["seed"])))
    log = pd.DataFrame({k: np.asarray(v) for k, v in log.items()})
    sw = switching_table(th_sum, win_n, CONFIG)

    # attach each cycle's measures to the first logged step >= its end
    sw_at = sw.copy()
    sw_at["step"] = [log["step"][log["step"] >= s].min() for s in sw_at["cycle_end_step"]]
    df = log.merge(sw_at.drop(columns="cycle_end_step"), on="step", how="left")
    cols = ["step", "train_accuracy", "test_accuracy", "train_loss", "test_loss",
            "focus_class", "focus_weight", "osc_phase",
            "cycle", "switch_frac", "switch_persist", "switching_distance"]
    df = df[cols]
    df["cycle"] = df["cycle"].astype("Int64")
    csv_path = OUT / f"{name}_A{A:g}_T{T_}_seed{CONFIG['seed']}.csv"
    df.to_csv(csv_path, index=False)

    # first evaluation with zero training error (only used as a marker in the plots)
    perfect = log.index[log.train_accuracy >= 1.0]
    first_perfect = int(log.step[perfect[0]]) if len(perfect) else None

    RESULTS[name] = dict(A=A, params=params, log=log, sw=sw, first_perfect=first_perfect, csv=csv_path)
    print(f"{name} (A={A:g}): {time.time() - t0:5.1f}s | first 100% train acc at step {first_perfect} | "
          f"final train acc = {log.train_accuracy.iloc[-1]:.4f}, test acc = {log.test_accuracy.iloc[-1]:.4f} | "
          f"{len(sw)} cycles | -> {csv_path}")

pd.read_csv(RESULTS["DB"]["csv"]).dropna(subset=["switch_frac"]).head()

## 7 · Figures
Shared style for the training figures. The dotted vertical line marks the first evaluation with 100 % training accuracy.

In [ ]:
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "font.size": 12, "axes.titlesize": 13,
    "axes.titleweight": "bold", "axes.labelsize": 12, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "grid.linestyle": "-",
    "legend.frameon": False, "xtick.direction": "out", "ytick.direction": "out",
})
TRAIN_C, TEST_C = "#1d3557", "#e76f51"
RUN_COLORS = {"UB": "#E69F00", "DB": "#4DAF4A"}           # protocol colours, used everywhere
CLASS_COLORS = ["#d7191c", "#808080", "#2b83ba"]          # "red_grey_blue"
TITLES = {"UB": "Uniform Batching (A = {A:g})", "DB": "Dynamical Batching (A = {A:g}, T = %d)" % T_}


def decorate(ax, r):
    s = r["first_perfect"]
    if s is not None:
        ax.axvline(s, color="0.35", ls=":", lw=1.4, zorder=1)
        ax.annotate("100% train", (s, 1), xycoords=("data", "axes fraction"), xytext=(4, -14),
                    textcoords="offset points", color="0.3", fontsize=10, zorder=10,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85))
    ax.set_xlim(0, r["log"]["step"].iloc[-1])
    ax.set_xlabel("Training step")


LEGEND = [Line2D([], [], color=TRAIN_C, lw=2, label="Train"), Line2D([], [], color=TEST_C, lw=2, label="Test")]

### 7.1 · Batch composition (dynamics)
Fraction of each class in the batch, $\lfloor p_c B\rfloor / \sum_{c'} \lfloor p_{c'} B\rfloor$, computed with exactly the same
formula as the training loop, drawn as stacked bands (they always sum to 1). Only the first `N_CYCLES_SHOWN` cycles are
drawn (the pattern then repeats identically); thin white lines mark oscillation boundaries, dashed lines cycle boundaries,
dotted lines the uniform split. Under UB ($A=1$) every class sits at $1/C$;
under DB each class takes its turn with a $1 \to A \to 1$ oscillation.


In [ ]:
N_CYCLES_SHOWN = 3


def batch_composition(A, steps, cfg):
    C = cfg["data"]["num_classes"]; P = cfg["oscillations"]["period_length"]; B = cfg["training"]["batch_size"]
    A = np.float32(A); ps = (steps % P).astype(np.float32); focus = (steps // P) % C
    slope = np.float32(2.0) * (A - 1) / np.float32(P)
    fw = np.where(ps < P / 2.0, 1 + ps * slope, 2 * A - ps * slope - 1).astype(np.float32)
    w = np.ones((steps.size, C), np.float32); w[np.arange(steps.size), focus] = fw
    counts = np.floor(w / w.sum(1, keepdims=True) * B)
    return counts / counts.sum(1, keepdims=True)


steps = np.arange(N_CYCLES_SHOWN * C_ * T_)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
for ax, (name, r) in zip(axes, RESULTS.items()):
    frac = batch_composition(r["A"], steps, CONFIG)
    for k in range(1, N_CYCLES_SHOWN * C_):
        ax.axvline(k * T_, color="white" if k % C_ else "0.15", lw=0.8 if k % C_ else 1.6,
                   ls="-" if k % C_ else "--", zorder=2)
    ax.stackplot(steps, frac.T, colors=CLASS_COLORS, alpha=0.85, edgecolor="white", linewidth=0.6, zorder=1)
    for k in range(1, C_):   # uniform reference: the band edges under UB
        ax.axhline(k / C_, color="0.2", ls=":", lw=0.9, zorder=3)
    ax.set_title(TITLES[name].format(A=r["A"]), color=RUN_COLORS[name])
    ax.set_xlim(0, steps[-1] + 1); ax.set_ylim(0, 1.0)
    ax.set_xlabel("Training step"); ax.grid(False)
axes[0].set_ylabel("Fraction of the batch\n(stacked)")
fig.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.85, label=f"Class {i}") for i, c in enumerate(CLASS_COLORS)],
           loc="upper center", ncol=C_, bbox_to_anchor=(0.5, 1.08))
fig.tight_layout(); fig.savefig(OUT / "batch_composition.png", bbox_inches="tight"); plt.show()

### 7.2 · Training / test accuracy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, (name, r) in zip(axes, RESULTS.items()):
    decorate(ax, r)
    ax.plot(r["log"].step, r["log"].test_accuracy, color=TEST_C, lw=2.0, alpha=0.85)
    ax.plot(r["log"].step, r["log"].train_accuracy, color=TRAIN_C, lw=1.1)
    ax.set_title(TITLES[name].format(A=r["A"]), color=RUN_COLORS[name])
    ax.set_ylim(min(0.3, r["log"].test_accuracy.min() - 0.02), 1.01)
axes[0].set_ylabel("Accuracy")
fig.legend(handles=LEGEND, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.08))
fig.tight_layout(); fig.savefig(OUT / "accuracy.png", bbox_inches="tight"); plt.show()

### 7.3 · Loss

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, (name, r) in zip(axes, RESULTS.items()):
    decorate(ax, r)
    ax.plot(r["log"].step, r["log"].test_loss, color=TEST_C, lw=2.0, alpha=0.85)
    ax.plot(r["log"].step, r["log"].train_loss, color=TRAIN_C, lw=1.1)
    ax.set_yscale("log"); ax.set_title(TITLES[name].format(A=r["A"]), color=RUN_COLORS[name])
axes[0].set_ylabel("Cross-entropy loss")
fig.legend(handles=LEGEND, loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.08))
fig.tight_layout(); fig.savefig(OUT / "loss.png", bbox_inches="tight"); plt.show()

### 7.4 · Decision boundaries (final parameters)
UB left, DB right. Region colours = predicted class; training samples are dots, test samples are crosses,
both coloured by their true class.

In [ ]:
cmap = ListedColormap(CLASS_COLORS, name="red_grey_blue")
predict = jax.jit(lambda p, x: jnp.argmax(apply_fn(p, x), axis=1))

@jax.jit
def margins(p, x):
    # logit_i - max_{j != i} logit_j: its zero level is class i's boundary (smooth, unlike argmax pixels)
    z = apply_fn(p, x)
    others = jnp.where(jnp.eye(z.shape[1], dtype=bool)[None], -jnp.inf, z[:, None, :]).max(-1)
    return z - others

lim = 1.12 * float(np.abs(np.r_[x_train, x_test]).max())
g = np.linspace(-lim, lim, 500, dtype=np.float32)
gx, gy = np.meshgrid(g, g)
grid = np.c_[gx.ravel(), gy.ravel()]
point_colors = lambda ys: [CLASS_COLORS[k] for k in ys]

fig, axes = plt.subplots(1, 2, figsize=(12, 6.3))
for ax, (name, r) in zip(axes, RESULTS.items()):
    zz = np.asarray(predict(r["params"], grid)).reshape(gx.shape)
    mg = np.asarray(margins(r["params"], grid)).reshape(*gx.shape, C_)
    ax.imshow(zz, extent=(-lim, lim, -lim, lim), origin="lower", cmap=cmap, alpha=0.22,
              vmin=-0.5, vmax=C_ - 0.5, interpolation="nearest", zorder=0)
    for c in range(C_):
        ax.contour(gx, gy, mg[..., c], levels=[0.0], colors="0.15", linewidths=1.1, zorder=1)
    ax.scatter(x_train[:, 0], x_train[:, 1], c=point_colors(y_train), marker="o", s=22,
               edgecolors="white", linewidths=0.5, zorder=3)
    ax.scatter(x_test[:, 0], x_test[:, 1], c=point_colors(y_test), marker="x", s=26, linewidths=1.3, zorder=2)
    acc_tr = float(np.mean(np.asarray(predict(r["params"], x_train)) == y_train))
    acc_te = float(np.mean(np.asarray(predict(r["params"], x_test)) == y_test))
    ax.set_title(f"{TITLES[name].format(A=r['A'])}\ntrain acc = {acc_tr:.4f}  ·  test acc = {acc_te:.4f}",
                 color=RUN_COLORS[name])
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_color(RUN_COLORS[name]); sp.set_linewidth(2.0)
handles = [Line2D([], [], marker="s", ls="", markersize=10, markerfacecolor=c, markeredgecolor="none",
                  label=f"Class {i}") for i, c in enumerate(CLASS_COLORS)]
handles += [Line2D([], [], marker="o", ls="", markersize=7, markerfacecolor="0.4", markeredgecolor="white", label="Train"),
            Line2D([], [], marker="x", ls="", markersize=7, markeredgecolor="0.4", markeredgewidth=1.5, label="Test")]
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.legend(handles=handles, loc="lower center", ncol=len(handles), bbox_to_anchor=(0.5, 0.0), columnspacing=1.8)
fig.savefig(OUT / "decision_boundaries.png", bbox_inches="tight"); plt.show()

### 7.5 · Switching measures — panels (a)–(c), single seed
(a) switching-set size $|\mathcal S|/n$, (b) persistence $J(\mathcal S_c,\mathcal S_{c-1})$, (c) pair-averaged switching
distance $d_{ij}$, all against the cycle index. UB dashed, DB solid. Colours, line widths and fonts are set in the
first lines of the cell.

In [ ]:
# ---- style constants --------------------------------------------------------
UB_COLOR, DB_COLOR = RUN_COLORS["UB"], RUN_COLORS["DB"]
UB_LABEL, DB_LABEL = "Uniform", "Dynamical Batching"
LW_UB, LW_DB = 2.0, 2.4
UB_MARKERS = False
RC = {"font.size": 15, "axes.labelsize": 15, "xtick.labelsize": 13, "ytick.labelsize": 13,
      "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3}
FIG_W = 11.6
LABELS = {
    "size": "Switching-set size\n$|\\mathcal{S}|/n$",
    "persist": "Switching-set persistence\n$J(\\mathcal{S}_c,\\mathcal{S}_{c-1})$",
    "dist": "Switching distance $d_{ij}$",
}
# -----------------------------------------------------------------------------

def curve(sw, col, max_cycle):
    s = sw[sw.cycle <= max_cycle].dropna(subset=[col])
    return s.cycle.to_numpy(), s[col].to_numpy()

def draw(ax, cv, dynamical, markers=True):
    x, y = cv
    ax.plot(x, y, color=DB_COLOR if dynamical else UB_COLOR, ls="-" if dynamical else "--",
            lw=LW_DB if dynamical else LW_UB, marker="o" if markers else None, markersize=4)

def top_of(*cvs, pad=1.1):
    tops = [np.nanmax(y) for _, y in cvs if len(y) and np.isfinite(y).any()]
    return pad * max(tops) if tops else 1.0

def cycle_axis(ax, max_cycle):
    ax.set_xlim(0, max_cycle); ax.set_xlabel("Cycle")

def panel_label(ax, letter, dx=-62, dy=14):
    ax.annotate(f"({letter})", xy=(0, 1), xycoords="axes fraction", xytext=(dx, dy),
                textcoords="offset points", fontsize=18, fontweight="bold", ha="left", va="bottom")


ub, db = RESULTS["UB"]["sw"], RESULTS["DB"]["sw"]
max_cycle = int(max(ub.cycle.max(), db.cycle.max()))

with plt.rc_context(RC):
    fig = plt.figure(figsize=(FIG_W, 4.4))
    gs = fig.add_gridspec(1, 3, left=0.085, right=0.99, bottom=0.16, top=0.8, wspace=0.55)
    axes = [fig.add_subplot(gs[0, i]) for i in range(3)]

    ax = axes[0]
    cu, cd = curve(ub, "switch_frac", max_cycle), curve(db, "switch_frac", max_cycle)
    draw(ax, cu, dynamical=False, markers=UB_MARKERS); draw(ax, cd, dynamical=True, markers=False)
    ax.set_ylim(0, top_of(cu, cd, pad=1.15)); ax.set_ylabel(LABELS["size"])

    ax = axes[1]
    draw(ax, curve(ub, "switch_persist", max_cycle), dynamical=False, markers=UB_MARKERS)
    draw(ax, curve(db, "switch_persist", max_cycle), dynamical=True, markers=False)
    ax.set_ylim(0, 1.0); ax.set_ylabel(LABELS["persist"])

    ax = axes[2]
    cu, cd = curve(ub, "switching_distance", max_cycle), curve(db, "switching_distance", max_cycle)
    draw(ax, cu, dynamical=False, markers=UB_MARKERS); draw(ax, cd, dynamical=True, markers=False)
    ax.set_ylim(0, top_of(cu, cd)); ax.set_ylabel(LABELS["dist"])

    for ax, letter in zip(axes, "abc"):
        cycle_axis(ax, max_cycle); panel_label(ax, letter)

    handles = [Line2D([], [], color=UB_COLOR, ls="--", lw=LW_UB, label=UB_LABEL),
               Line2D([], [], color=DB_COLOR, ls="-", lw=LW_DB, label=DB_LABEL)]
    fig.legend(handles=handles, loc="upper center", ncol=2, handlelength=2.4, columnspacing=3.8,
               handletextpad=0.7, borderaxespad=0.0, bbox_to_anchor=(0.52, 0.99), fontsize=18, frameon=False)
    fig.savefig(OUT / "switching_abc.png", bbox_inches="tight"); fig.savefig(OUT / "switching_abc.pdf", bbox_inches="tight")
    plt.show()

## 8 · Download everything

In [ ]:
import shutil
shutil.make_archive("results", "zip", OUT)
print(sorted(p.name for p in OUT.iterdir()))
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    print("Not on Colab: results are in", OUT.resolve())